In [3]:
from gitsource import GithubRepositoryDataReader

reader = GithubRepositoryDataReader(
    repo_owner="DataTalksClub",
    repo_name="llm-zoomcamp",
    commit_id="8c1834d",
    allowed_extensions={"md"},
    filename_filter=lambda path: "/lessons/" in path,
)
documents = [file.parse() for file in reader.read()]
print(len(documents))   # should print 72

72


In [5]:
import json
from pydantic import BaseModel
from openai import OpenAI
from evaluation_utils import llm_structured

openai_client = OpenAI()

class Questions(BaseModel):
    questions: list[str]

target_files = {
    "01-agentic-rag/lessons/01-intro.md",
    "01-agentic-rag/lessons/02-environment.md",
    "01-agentic-rag/lessons/03-rag.md",
}
first_three = [d for d in documents if d["filename"] in target_files]

data_gen_instructions = """
You emulate a student who is taking our LLM course.
You are given one lesson page from the course.
Formulate 5 questions this student might ask that are answered by this page.

Rules:
- The page should contain the answer to each question.
- Make the questions complete and not too short.
- Use as few words as possible from the page; don't copy its phrasing.
- The questions should resemble how people actually ask things online:
  not too formal, not too short, not too long.
- Ask about the content of the lesson, not about its formatting or filename.
""".strip()

input_tokens = []
all_qa_records = []

for doc in first_three:
    user_prompt = json.dumps(doc)
    result, usage = llm_structured(
        openai_client,
        data_gen_instructions,
        user_prompt,
        Questions
    )
    input_tokens.append(usage.input_tokens)
    for q in result.questions:
        all_qa_records.append({"question": q, "filename": doc["filename"]})

print(sum(input_tokens) / len(input_tokens))   # Q1 answer

1353.0


In [6]:
import pandas as pd

gt_df = pd.read_csv("ground-truth.csv")
ground_truth = gt_df.to_dict(orient="records")
print(len(ground_truth))   # should print 360

360


In [7]:
from gitsource import chunk_documents
from minsearch import Index, VectorSearch
from embedder import Embedder
import numpy as np

chunks = chunk_documents(documents, size=2000, step=1000)
print(len(chunks))   # should print 295

embedder = Embedder("models/Xenova/all-MiniLM-L6-v2")
vectors = embedder.encode_batch([c["content"] for c in chunks])
X = np.array(vectors)

text_idx = Index(text_fields=["content"], keyword_fields=[])
text_idx.fit(chunks)

vector_idx = VectorSearch()
vector_idx.fit(X, chunks)

def text_search(query, num_results=5):
    return text_idx.search(query, num_results=num_results)

def vector_search(query, num_results=5):
    v = embedder.encode(query)
    return vector_idx.search(v, num_results=num_results)

295


In [8]:
q = ground_truth[0]["question"]

print(text_search(q)[0]["filename"])     # Q2 answer
print(vector_search(q)[0]["filename"])   # Q3 answer

01-agentic-rag/lessons/03-rag.md
01-agentic-rag/lessons/01-intro.md
